# Equip Triage · joint log and memo encoder
장비 로그와 한·영 관측 메모에서 첫 점검, 후보별 상태, 근거 원본 행을 평가합니다. 모든 작업 기록은 합성입니다. 아래 기본 경로는 CUDA 본학습용입니다. 소스는 노트북에 내장되어 있습니다. 공개 다국어 비교 모델은 해당 셀에서 고정된 원본을 내려받습니다.

In [ ]:
from pathlib import Path
import base64, hashlib, io, json, subprocess, sys, zipfile
import torch
print("PyTorch", torch.__version__, "CUDA", torch.version.cuda, "available", torch.cuda.is_available())
assert torch.cuda.is_available(), "GPU 런타임에 연결하세요. 이 본학습 노트북은 CUDA를 요구합니다."
print("GPU", torch.cuda.get_device_name(0))
ROOT = Path("/content/equip-triage-joint-v1")
mounted = Path("/content/drive/MyDrive")
STORE = (mounted if mounted.is_dir() else Path("/content")) / "EquipTriage-joint-v1-runs"
WORK = STORE / "main"
PUBLIC = Path("/content/equip-joint-public-base")
SEED = 17
print("Checkpoint directory", WORK, "persistent mounted Drive", mounted.is_dir())

## 소스 복원과 설치
체크포인트 폴더를 유지한 상태에서 같은 셀을 다시 실행하면 저장된 optimizer 단계 이후부터 이어갑니다. 새 VM으로 옮길 때는 그 폴더 전체를 개인 저장소에 복사해 두세요. 로그인, 드라이브 연결 및 유료 GPU 구매는 필수 단계가 아닙니다.

In [ ]:
archive = base64.b64decode('...')
assert hashlib.sha256(archive).hexdigest() == '045b3c10a030971a7720da26d2ff269a484b0c776c1b2a8b677fe609f7cd2dbb'
ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(archive)) as z:
    z.extractall(ROOT)
print("Verified source archive", len(archive), "bytes")

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn==1.8.0", "transformers==4.57.3", "safetensors>=0.5,<1"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "--no-build-isolation", str(ROOT)], check=True)
def run(*args):
    subprocess.run([sys.executable, "-B", "-u", "-m", "research.joint_encoder.run",
                    "--work", str(WORK), "--profile", "main", "--public-assets", str(PUBLIC),
                    *map(str, args)], cwd=ROOT, check=True)

## 자료와 시점 검증
판단 시점 이후·타 LOT·사후 조치가 관측에 들어오지 않는지, 로그만 또는 메모만 바꾼 쌍이 분리되는지 먼저 검증합니다. 재생성한 세 분할의 해시가 고정값과 달라지면 여기서 멈춥니다.

In [ ]:
run("prepare")
manifest = json.loads((WORK / "manifest.json").read_text())
expected = {'test': '7b982fe68524ed9034d5990767370b09cf1710a2b6e1ff7a82a401bf445cfb36', 'train': 'ef4254e2e652158d1ac93625c7affe438841d367c314e29a00a4d70fc132675f', 'val': '730a3265489087fa30b7f610ea83ef57945f9157e38dc87442bae20b7e2a3e6a'}
assert {k:v["labeled_sha256"] for k,v in manifest["data"].items()} == expected
print({k:(v["rows"], len(v["families"])) for k,v in manifest["data"].items()})

## 자체 모델과 대조군
첫 셀은 자체 결합 모델을 먼저 학습합니다. 이어지는 셀은 같은 분할에서 규칙·선형 기준선, 직접 선택, 모달리티 제거 및 동일 규모의 구조 비교를 학습합니다. 한 단계는 메모리가 허용하는 마이크로배치 네 건씩 누적한 16건입니다.

In [ ]:
def fit(arm):
    run("fit", "--arm", arm, "--device", "cuda", "--seed", SEED,
        "--microbatch", 4, "--save-every", 100)
fit("joint")
print(json.loads((WORK / "models" / f"joint-{SEED}" / "fit.json").read_text())["progress"])

In [ ]:
run("fit-baselines")
for arm in ("pooled", "parallel", "unbound", "log_only", "memo_only", "direct"):
    fit(arm)

## 공개 한·영 사전학습 비교군
다국어 DistilBERT의 공식 모델·토크나이저 다섯 파일을 고정 SHA-256으로 확인합니다. 같은 전체 관측 문장, 메타데이터, 상태·근거 헤드, 학습 횟수 및 보류 정책을 적용합니다. 체크포인트와 AdamW 상태를 저장할 여유 공간이 필요합니다.

In [ ]:
run("fetch-public")
fit("public")

## 동결 평가
모든 비교군의 학습·검증 임계값이 저장되어야 최종 평가가 열립니다. 최종 언어별 macro-F1, 보류 오해제, 근거 행 적중 및 계열 단위 구간을 같은 평가기로 집계합니다. 41, 73은 사전 명시한 반복 seed이며 추가로 실행할 때 같은 프로토콜을 유지합니다.

In [ ]:
report = WORK / f"evaluation-{SEED}.json"
if not report.exists():
    run("evaluate", "--device", "cuda", "--seed", SEED)
result = json.loads(report.read_text())
print("Comparator", result["comparator"], "point pass", result["point_pass"],
      "interval pass", result["interval_pass"])
for arm, outputs in result["results"].items():
    print(arm, [(lang, round(outputs["selected"]["language/"+lang]["macro_f1"], 4),
                 outputs["selected"]["language/"+lang]["false_releases"]) for lang in ("en","ko")])
print("Full report", report)